In [1]:
import os
import json
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [2]:
def load_sweep_results_with_hyperparams(sweep_ids: list, wandb_dir: Path) -> pd.DataFrame:
    """Load all run results including hyperparameters from multiple sweeps."""
    all_results = []
    
    for sweep_id in sweep_ids:
        sweep_dir = wandb_dir / f"sweep-{sweep_id}"
        if not sweep_dir.exists():
            print(f"Warning: Sweep directory not found for {sweep_id}")
            continue
            
        print(f"Loading sweep: {sweep_id}")
        
        # Get all run IDs from sweep config files
        for config_file in sweep_dir.glob("config-*.yaml"):
            run_id = config_file.stem.replace("config-", "")
            
            # Find the corresponding run directory
            run_dirs = list(wandb_dir.glob(f"run-*-{run_id}"))
            if not run_dirs:
                continue
            
            run_dir = run_dirs[0]
            
            # Load config from sweep directory
            with open(config_file) as f:
                config = yaml.safe_load(f)
            
            # Load summary from run directory
            summary_file = run_dir / "files" / "wandb-summary.json"
            if not summary_file.exists():
                continue
                
            with open(summary_file) as f:
                summary = json.load(f)
            
            # Helper to extract value from config
            def get_config_value(key):
                val = config.get(key, {})
                if isinstance(val, dict):
                    return val.get("value")
                return val
            
            # Extract all relevant info including hyperparameters
            result = {
                "sweep_id": sweep_id,
                "run_id": run_id,
                "setting": get_config_value("setting"),
                "method": get_config_value("method"),
                "seed": get_config_value("seed"),
                "lr": get_config_value("lr"),
                "beta_efc": get_config_value("beta_efc"),
                "target_lr": get_config_value("target_lr"),
                "batch_size": get_config_value("batch_size"),
                "epochs": get_config_value("epochs"),
                "final_avg_accuracy": summary.get("final_avg_accuracy"),
                "final_forgetting": summary.get("final_forgetting"),
                "layer_size": get_config_value("layer_size"),
                "cnn_pretrained": get_config_value("cnn_pretrained")
            }
            all_results.append(result)
    
    return pd.DataFrame(all_results)

In [5]:
# Configure your sweep IDs here (can be a single ID or a list)

SWEEP_DICTS = {"taskIL_EFC_MNIST": "r1mbnfcc",
               "taskIL_EFC_CIFAR10": "ish4fsaz",
               "taskIL_EFC_MNIST_Encoded": "48h482zz",
               "taskIL_EFC_TinyImageNet": "2y74n3t7", 
               
               "classIL_EFC_MNIST": "3z7o6b9q",
               "classIL_EFC_CIFAR10": "5k1j6g9y",
               "classIL_EFC_MNIST_Encoded": "a9j3x5o3",
               "classIL_EFC_TinyImageNet": "b6j9w4p0"}

WANDB_DIR = Path("./wandb")

wanted_sweep = "taskIL_EFC_TinyImageNet"  # Change this to select different sweeps
SWEEP_ID = [SWEEP_DICTS[wanted_sweep]]


# Load all results from all sweeps
df = load_sweep_results_with_hyperparams(SWEEP_ID, WANDB_DIR)

Loading sweep: 2y74n3t7


In [12]:
# Summary statistics
avg_acc = df['final_avg_accuracy'].mean()
std_acc = df['final_avg_accuracy'].std()
n_seeds = df['seed'].nunique()

print(f"=== Summary for {wanted_sweep} ===")
print(f"Average Accuracy: {avg_acc:.2f}% ± {std_acc:.2f}%")
print(f"Number of seeds: {n_seeds}")
print()

# Performance of all models in the sweep
print(f"=== Individual Run Performance ===")
performance_df = df[['seed', 'method', 'setting', 'lr', 'beta_efc', 'target_lr', 'final_avg_accuracy']].copy()
performance_df = performance_df.sort_values('seed')
performance_df['final_avg_accuracy'] = performance_df['final_avg_accuracy'].round(2)
print(performance_df.to_string(index=False))

=== Summary for taskIL_EFC_TinyImageNet ===
Average Accuracy: 28.96% ± 0.84%
Number of seeds: 10

=== Individual Run Performance ===
 seed method            setting     lr  beta_efc  target_lr  final_avg_accuracy
    0    efc TaskILTinyImageNet 0.0001       0.1        0.1               27.55
    1    efc TaskILTinyImageNet 0.0001       0.1        0.1               29.71
    2    efc TaskILTinyImageNet 0.0001       0.1        0.1               28.29
    3    efc TaskILTinyImageNet 0.0001       0.1        0.1               29.91
    4    efc TaskILTinyImageNet 0.0001       0.1        0.1               27.83
    5    efc TaskILTinyImageNet 0.0001       0.1        0.1               28.66
    6    efc TaskILTinyImageNet 0.0001       0.1        0.1               29.50
    7    efc TaskILTinyImageNet 0.0001       0.1        0.1               29.32
    8    efc TaskILTinyImageNet 0.0001       0.1        0.1               29.07
    9    efc TaskILTinyImageNet 0.0001       0.1        0.1        

## Other Methods 

In [8]:
# Dictionary mapping sweep names to their IDs for "other" methods
OTHER_SWEEP_DICTS = {
    "classIL_other_CIFAR10": "1a8wifwl",
    "taskIL_other_MNIST": "r0ud5v5c",
    "taskIL_other_CIFAR10": "691lbl0s",
    "taskIL_other_MNIST_Encoded": "4m769eeo",
    "taskIL_other_TinyImageNet": "wrwbc2ql",
    "classIL_other_MNIST": "anrjyvi5",
    "classIL_other_MNIST_Encoded": "nygl2707",
    "classIL_other_TinyImageNet": "4astdr0c",
}


for sweep in OTHER_SWEEP_DICTS.keys():
    

# Load results for a specific sweep
    wanted_sweep = sweep
    df = load_sweep_results_with_hyperparams([OTHER_SWEEP_DICTS[wanted_sweep]], WANDB_DIR)

    # Summary statistics grouped by method
    print(f"=== Summary for {wanted_sweep} ===\n")
    summary = df.groupby('method').agg({
        'final_avg_accuracy': ['mean', 'std', 'count'],
    }).round(2)
    summary.columns = ['Acc Mean', 'Acc Std', 'N Seeds']
    print(summary.to_string())

    print(f"\n=== Individual Run Performance ===")
    # performance_df = df[['seed', 'method', 'setting', 'final_avg_accuracy']].copy()
    # performance_df = performance_df.sort_values(['method', 'seed'])
    # performance_df['final_avg_accuracy'] = performance_df['final_avg_accuracy'].round(2)
    # print(performance_df.to_string(index=False))


Loading sweep: 1a8wifwl
=== Summary for classIL_other_CIFAR10 ===

        Acc Mean  Acc Std  N Seeds
method                            
bp         19.70     0.02        5
ewc        19.81     0.08        5
oewc       19.80     0.05        5
si           NaN      NaN        0

=== Individual Run Performance ===
Loading sweep: r0ud5v5c
=== Summary for taskIL_other_MNIST ===

        Acc Mean  Acc Std  N Seeds
method                            
bp         95.52     2.08        5
ewc        96.05     1.45        5
oewc       96.86     1.74        5
si           NaN      NaN        0

=== Individual Run Performance ===
Loading sweep: 691lbl0s
=== Summary for taskIL_other_CIFAR10 ===

        Acc Mean  Acc Std  N Seeds
method                            
bp         93.20     2.11        5
ewc        90.61     3.75        5
oewc       94.15     1.62        5
si           NaN      NaN        0

=== Individual Run Performance ===
Loading sweep: 4m769eeo
=== Summary for taskIL_other_MNIST_Encode

In [9]:
# Dictionary mapping sweep names to their IDs for "other" methods
OTHER_SWEEP_DICTS = {
    "classIL_other_CIFAR10": "1a8wifwl",
    "taskIL_other_MNIST": "r0ud5v5c",
    "taskIL_other_CIFAR10": "691lbl0s",
    "taskIL_other_MNIST_Encoded": "4m769eeo",
    "taskIL_other_TinyImageNet": "wrwbc2ql",
    "classIL_other_MNIST": "anrjyvi5",
    "classIL_other_MNIST_Encoded": "nygl2707",
    "classIL_other_TinyImageNet": "4astdr0c",
}

# Column order for CSV
columns = [
    "Task IL MNIST", "Task IL Encoded MNIST", "Task IL CIFAR10", "Task IL Tiny Image Net",
    "Class IL MNIST", "Class IL MNIST Encoded", "Class IL CIFAR10", "Class IL Tiny Image Net"
]

# Map sweep names to column names
sweep_to_col = {
    "taskIL_other_MNIST": "Task IL MNIST",
    "taskIL_other_MNIST_Encoded": "Task IL Encoded MNIST",
    "taskIL_other_CIFAR10": "Task IL CIFAR10",
    "taskIL_other_TinyImageNet": "Task IL Tiny Image Net",
    "classIL_other_MNIST": "Class IL MNIST",
    "classIL_other_MNIST_Encoded": "Class IL MNIST Encoded",
    "classIL_other_CIFAR10": "Class IL CIFAR10",
    "classIL_other_TinyImageNet": "Class IL Tiny Image Net",
}

# Initialize results dataframe
methods = ["BP", "EWC", "oEWC", "SI"]
results_df = pd.DataFrame(index=methods, columns=columns)

# Load each sweep and populate the table
for sweep_name, sweep_id in OTHER_SWEEP_DICTS.items():
    if sweep_id == "xxx":
        continue  # skip unfilled sweeps
    
    df = load_sweep_results_with_hyperparams([sweep_id], WANDB_DIR)
    col_name = sweep_to_col[sweep_name]
    
    for method in df['method'].unique():
        method_df = df[df['method'] == method]
        mean_acc = method_df['final_avg_accuracy'].mean()
        std_acc = method_df['final_avg_accuracy'].std()
        
        # Map method name to row name
        row_name = {"bp": "BP", "ewc": "EWC", "oewc": "oEWC", "si": "SI"}[method]
        results_df.loc[row_name, col_name] = f"{mean_acc:.2f} ± {std_acc:.2f}"

print(results_df.to_string())

# Save to CSV
results_df.to_csv("other_methods_results.csv")
print("\nSaved to other_methods_results.csv")


Loading sweep: 1a8wifwl
Loading sweep: r0ud5v5c
Loading sweep: 691lbl0s
Loading sweep: 4m769eeo
Loading sweep: wrwbc2ql
Loading sweep: anrjyvi5
Loading sweep: nygl2707
Loading sweep: 4astdr0c
     Task IL MNIST Task IL Encoded MNIST Task IL CIFAR10 Task IL Tiny Image Net Class IL MNIST Class IL MNIST Encoded Class IL CIFAR10 Class IL Tiny Image Net
BP    95.52 ± 2.08          90.70 ± 5.62    93.20 ± 2.11           34.62 ± 0.54   19.65 ± 0.14           19.48 ± 0.05     19.70 ± 0.02             4.55 ± 0.04
EWC   96.05 ± 1.45          90.95 ± 5.76    90.61 ± 3.75           34.15 ± 0.40   19.54 ± 0.03           19.46 ± 0.02     19.81 ± 0.08             4.41 ± 0.14
oEWC  96.86 ± 1.74          97.63 ± 1.36    94.15 ± 1.62           35.89 ± 0.35   19.57 ± 0.05           19.42 ± 0.02     19.80 ± 0.05             4.72 ± 0.04
SI       nan ± nan             nan ± nan       nan ± nan              nan ± nan      nan ± nan              nan ± nan        nan ± nan               nan ± nan

Saved to oth